In [13]:
!pip install chromadb sentence-transformers groq pandas

In [14]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq

In [15]:
groq_client = Groq(
    api_key="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"
)

In [16]:
df = pd.read_csv("college_notes.csv")

print("Dataset Loaded Successfully")
print("Total Rows:", len(df))

df.head()

Dataset Loaded Successfully
Total Rows: 15


,note_id,subject,topic,content
0,1,Data Engineering,ETL Pipelines,"ETL stands for Extract, Transform, Load. It is..."
1,2,Data Engineering,Data Warehousing,A data warehouse is a central repository that ...
2,3,Data Engineering,Apache Spark,Apache Spark is an open-source distributed com...
3,4,Data Engineering,Medallion Architecture,Medallion Architecture is a data design patter...
4,5,Data Engineering,Data Pipelines,A data pipeline is a series of automated steps...


In [17]:
documents = []

for _, row in df.iterrows():

    doc = " ".join(
        [str(value) for value in row.values if pd.notna(value)]
    )

    documents.append(doc)

print("Total Documents:", len(documents))

Total Documents: 15


In [18]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded


In [19]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embeddings Generated")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Generated


In [20]:
client = chromadb.PersistentClient(
    path="./college_db"
)

collection = client.get_or_create_collection(
    name="college_notes"
)

print("ChromaDB Collection Created")

ChromaDB Collection Created


In [21]:
for i, doc in enumerate(documents):

    collection.add(
        ids=[str(i)],
        documents=[doc],
        embeddings=[embeddings[i].tolist()]
    )

print("All Notes Indexed Successfully")

All Notes Indexed Successfully


In [22]:
def retrieve_relevant_chunks(question, top_k=3):

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]

In [23]:
question = "What is machine learning?"

results = retrieve_relevant_chunks(question)

for i, chunk in enumerate(results, 1):
    print(f"\nChunk {i}")
    print(chunk[:500])


Chunk 1

Subject: Machine Learning
Topic: Linear Regression
Content: Linear regression is a supervised machine learning algorithm used to predict a continuous numerical value. It finds the best-fit straight line through data points by minimizing the difference between predicted and actual values. The equation is y equals mx plus b, where m is the slope and b is the intercept. Linear regression is used in applications like predicting house prices, estimating student exam scores, and forecasting sales rev

Chunk 2

Subject: Machine Learning
Topic: Model Evaluation
Content: Model evaluation measures how well a machine learning model performs on unseen data. For regression models, common metrics include Mean Absolute Error which measures average prediction error, Root Mean Squared Error which penalizes large errors, and R-squared which measures the proportion of variance explained by the model. For classification models, common metrics include accuracy, precision, recall, and F1-score.




In [24]:
def generate_rag_answer(question, context):

    system_prompt = """
You are a helpful academic assistant.

RULES:
1. Answer only using the provided context.
2. If the answer is not found in the context, say exactly:
"I don't have enough information in my knowledge base to answer the question."
3. Do not use any external knowledge.
4. Keep answers clear and concise.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.1,
        max_tokens=500
    )

    return response.choices[0].message.content

In [29]:
def ask_college_assistant(question, top_k=3):

    print("\nQuestion:")
    print(question)

    print("\nRetrieving relevant notes...")

    retrieved_chunks = retrieve_relevant_chunks(
        question,
        top_k
    )

    print("\nTop Retrieved Chunks:")
    print("-" * 50)

    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"\nChunk {i}")
        print(chunk[:300])

    context = "\n\n".join(retrieved_chunks)

    answer = generate_rag_answer(
        question,
        context
    )

    print("\nAnswer:")
    print(answer)

    return answer

In [31]:
while True:

    question = input("\nAsk a Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    ask_college_assistant(question)


Ask a Question: python

Question:
python

Retrieving relevant notes...

Top Retrieved Chunks:
--------------------------------------------------

Chunk 1

Subject: Python
Topic: API Integration
Content: An API or Application Programming Interface allows different software systems to communicate with each other. In Python, the requests library is used to call REST APIs. A GET request retrieves data from a server. A POST request sends data to a server.

Chunk 2

Subject: Python
Topic: Pandas Library
Content: Pandas is a Python library for data manipulation and analysis. It provides two main data structures: Series for one-dimensional labeled data and DataFrame for two-dimensional tabular data. Key operations include reading data from CSV and JSON files, fi

Chunk 3

Subject: Data Engineering
Topic: Apache Spark
Content: Apache Spark is an open-source distributed computing framework designed for fast large-scale data processing. It processes data in memory instead of reading and writing 